# 01 Validate Safety Sign Dataset

Validate the current YOLO safety-sign dataset before handing it to a teammate for training.

This notebook is read-only for the dataset: it checks image-label pairs, class IDs, YOLO box values,
filename conventions, and class balance, then writes CSV reports under `reports/safety_signs_validation/`.


## Purpose

This notebook helps answer:

- Are all image-label pairs present?
- Are label rows valid YOLO format?
- Are class IDs limited to the four configured safety signs?
- Are negative samples (`NEG_001`, etc.) empty?
- Are mixed samples (`MIXED_001`, etc.) actually mixed?
- Are filenames consistent with their labels?
- How many objects/images exist per sign class?

It does not auto-label, rename, split, augment, train, or modify the dataset folders.


In [ ]:
from __future__ import annotations

from collections import Counter
import csv
from dataclasses import asdict, dataclass
from pathlib import Path
import sys
from typing import Iterable

try:
    import cv2
except ImportError:
    cv2 = None

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root(start: Path) -> Path:
    """Find this repo root from the current notebook directory."""
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "backend").exists() and (candidate / "frontend").exists():
            return candidate
    raise RuntimeError("Could not locate the ppe_labeling project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

SAFETY_SIGN_CLASSES = {
    0: "M014 Wear head protection",
    1: "M015 Wear high-visibility clothing",
    2: "P004 No thoroughfare",
    3: "W011 Slippery surface",
}

CLASS_CODES = {
    0: "M014",
    1: "M015",
    2: "P004",
    3: "W011",
}

IMAGES_DIR = PROJECT_ROOT / "data" / "safety_signs" / "images"
LABELS_DIR = PROJECT_ROOT / "data" / "safety_signs" / "labels"
REPORTS_DIR = PROJECT_ROOT / "reports" / "safety_signs_validation"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"Images: {IMAGES_DIR}")
print(f"Labels: {LABELS_DIR}")
print(f"Reports: {REPORTS_DIR}")
print(f"Allowed classes: {SAFETY_SIGN_CLASSES}")


## Validation Helpers


In [ ]:
@dataclass
class ValidationRow:
    base_name: str
    image_name: str
    label_name: str
    status: str
    errors: str
    warnings: str
    image_width: int | None
    image_height: int | None
    num_objects: int
    num_m014: int
    num_m015: int
    num_p004: int
    num_w011: int
    expected_prefix: str
    actual_prefix: str


def image_paths(images_dir: Path) -> list[Path]:
    if not images_dir.exists():
        return []
    return sorted(
        [path for path in images_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda path: path.name.lower(),
    )


def label_paths(labels_dir: Path) -> list[Path]:
    if not labels_dir.exists():
        return []
    return sorted(
        [path for path in labels_dir.iterdir() if path.is_file() and path.suffix.lower() == ".txt"],
        key=lambda path: path.name.lower(),
    )


def read_image_size(image_path: Path) -> tuple[int | None, int | None, list[str]]:
    warnings: list[str] = []
    if cv2 is None:
        warnings.append("OpenCV not available; image readability was not checked")
        return None, None, warnings

    image = cv2.imread(str(image_path))
    if image is None:
        return None, None, ["Image is unreadable by OpenCV"]

    height, width = image.shape[:2]
    return width, height, warnings


def parse_label_file(label_path: Path) -> tuple[list[tuple[int, float, float, float, float]], list[str], list[str]]:
    boxes: list[tuple[int, float, float, float, float]] = []
    errors: list[str] = []
    warnings: list[str] = []

    if not label_path.exists():
        warnings.append("Missing label file; treated as NEG candidate")
        return boxes, errors, warnings

    for line_number, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            errors.append(f"Line {line_number}: expected 5 YOLO values, got {len(parts)}")
            continue

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = [float(value) for value in parts[1:]]
        except ValueError:
            errors.append(f"Line {line_number}: class/box values are not numeric")
            continue

        if class_id not in SAFETY_SIGN_CLASSES:
            errors.append(f"Line {line_number}: unknown class id {class_id}")

        for field_name, value in (
            ("x_center", x_center),
            ("y_center", y_center),
            ("width", width),
            ("height", height),
        ):
            if value < 0 or value > 1:
                errors.append(f"Line {line_number}: {field_name}={value} outside [0, 1]")

        if width <= 0 or height <= 0:
            errors.append(f"Line {line_number}: width/height must be > 0")

        x1 = x_center - width / 2
        y1 = y_center - height / 2
        x2 = x_center + width / 2
        y2 = y_center + height / 2
        if x1 < -0.001 or y1 < -0.001 or x2 > 1.001 or y2 > 1.001:
            warnings.append(f"Line {line_number}: box extends outside image bounds")

        area = width * height
        if 0 < area < 0.0001:
            warnings.append(f"Line {line_number}: very small box area {area:.6f}")
        if area > 0.5:
            warnings.append(f"Line {line_number}: very large box area {area:.3f}")

        boxes.append((class_id, x_center, y_center, width, height))

    return boxes, errors, warnings


def actual_prefix(base_name: str) -> str:
    upper_name = base_name.upper()
    for prefix in ("M014", "M015", "P004", "W011", "MIXED", "NEG"):
        if upper_name == prefix or upper_name.startswith(f"{prefix}_"):
            return prefix
    return ""


def expected_prefix_for_boxes(boxes: Iterable[tuple[int, float, float, float, float]]) -> str:
    class_ids = sorted({box[0] for box in boxes if box[0] in SAFETY_SIGN_CLASSES})
    if not class_ids:
        return "NEG"
    if len(class_ids) == 1:
        return CLASS_CODES[class_ids[0]]
    return "MIXED"


def validate_one_sample(image_path: Path, label_path: Path) -> ValidationRow:
    errors: list[str] = []
    warnings: list[str] = []

    width: int | None = None
    height: int | None = None
    if image_path.exists():
        width, height, image_warnings = read_image_size(image_path)
        warnings.extend(image_warnings)
        if cv2 is not None and (width is None or height is None):
            errors.append("Image file is unreadable")
    else:
        errors.append("Missing image file")

    boxes, label_errors, label_warnings = parse_label_file(label_path)
    errors.extend(label_errors)
    warnings.extend(label_warnings)

    counts = Counter(box[0] for box in boxes)
    expected_prefix = expected_prefix_for_boxes(boxes)
    actual = actual_prefix(image_path.stem if image_path.exists() else label_path.stem)

    if actual and actual != expected_prefix:
        warnings.append(f"Filename prefix {actual} does not match labels; expected {expected_prefix}")
    if expected_prefix == "NEG" and boxes:
        errors.append("NEG sample has label rows")
    if actual == "NEG" and boxes:
        warnings.append("Filename says NEG but label file contains objects")
    if actual == "MIXED" and len({box[0] for box in boxes if box[0] in SAFETY_SIGN_CLASSES}) < 2:
        warnings.append("Filename says MIXED but fewer than two sign classes were found")

    return ValidationRow(
        base_name=image_path.stem if image_path.exists() else label_path.stem,
        image_name=image_path.name if image_path.exists() else "",
        label_name=label_path.name if label_path.exists() else "",
        status="invalid" if errors else "valid",
        errors="; ".join(dict.fromkeys(errors)),
        warnings="; ".join(dict.fromkeys(warnings)),
        image_width=width,
        image_height=height,
        num_objects=len(boxes),
        num_m014=counts[0],
        num_m015=counts[1],
        num_p004=counts[2],
        num_w011=counts[3],
        expected_prefix=expected_prefix,
        actual_prefix=actual,
    )


def validate_dataset(images_dir: Path, labels_dir: Path) -> list[ValidationRow]:
    images_by_stem = {path.stem: path for path in image_paths(images_dir)}
    labels_by_stem = {path.stem: path for path in label_paths(labels_dir)}
    all_stems = sorted(set(images_by_stem) | set(labels_by_stem), key=str.lower)

    rows: list[ValidationRow] = []
    for stem in all_stems:
        image_path = images_by_stem.get(stem, images_dir / f"{stem}.jpg")
        label_path = labels_by_stem.get(stem, labels_dir / f"{stem}.txt")
        rows.append(validate_one_sample(image_path, label_path))

    return rows


def write_rows_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return

    with path.open("w", encoding="utf-8", newline="") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def show_table(rows: list[dict], limit: int = 20) -> None:
    if pd is not None:
        display(pd.DataFrame(rows).head(limit))
    else:
        display(rows[:limit])


## Validate Dataset


In [ ]:
validation_rows = validate_dataset(IMAGES_DIR, LABELS_DIR)
validation_dicts = [asdict(row) for row in validation_rows]

validation_report_path = REPORTS_DIR / "validation_report.csv"
invalid_samples_path = REPORTS_DIR / "invalid_samples.csv"
warning_samples_path = REPORTS_DIR / "warning_samples.csv"

invalid_rows = [row for row in validation_dicts if row["status"] != "valid"]
warning_rows = [row for row in validation_dicts if row["warnings"]]

write_rows_csv(validation_report_path, validation_dicts)
write_rows_csv(invalid_samples_path, invalid_rows)
write_rows_csv(warning_samples_path, warning_rows)

print(f"Saved validation report: {validation_report_path}")
print(f"Saved invalid samples report: {invalid_samples_path}")
print(f"Saved warning samples report: {warning_samples_path}")
print(f"Rows checked: {len(validation_rows)}")
show_table(validation_dicts)


## Dataset Summary


In [ ]:
total_images = len(image_paths(IMAGES_DIR))
total_labels = len(label_paths(LABELS_DIR))
valid_count = sum(1 for row in validation_rows if row.status == "valid")
invalid_count = len(validation_rows) - valid_count
warning_count = sum(1 for row in validation_rows if row.warnings)

summary = {
    "total_images": total_images,
    "total_label_files": total_labels,
    "total_rows_checked": len(validation_rows),
    "valid_rows": valid_count,
    "invalid_rows": invalid_count,
    "warning_rows": warning_count,
    "total_objects": sum(row.num_objects for row in validation_rows),
    "total_m014": sum(row.num_m014 for row in validation_rows),
    "total_m015": sum(row.num_m015 for row in validation_rows),
    "total_p004": sum(row.num_p004 for row in validation_rows),
    "total_w011": sum(row.num_w011 for row in validation_rows),
    "negative_images": sum(1 for row in validation_rows if row.expected_prefix == "NEG"),
    "mixed_images": sum(1 for row in validation_rows if row.expected_prefix == "MIXED"),
}

summary_path = REPORTS_DIR / "dataset_summary.csv"
write_rows_csv(summary_path, [summary])

print(f"Saved dataset summary: {summary_path}")
show_table([summary])


## Class Counts


In [ ]:
class_count_rows = [
    {"class_id": class_id, "class_code": CLASS_CODES[class_id], "class_name": SAFETY_SIGN_CLASSES[class_id], "objects": summary[f"total_{CLASS_CODES[class_id].lower()}"]}
    for class_id in sorted(SAFETY_SIGN_CLASSES)
]

class_counts_path = REPORTS_DIR / "class_counts.csv"
write_rows_csv(class_counts_path, class_count_rows)

print(f"Saved class counts: {class_counts_path}")
show_table(class_count_rows)


## Review Problems Before Handoff


In [ ]:
if not IMAGES_DIR.exists():
    print(f"Missing image directory: {IMAGES_DIR}")
elif not validation_rows:
    print("No safety-sign samples found.")
else:
    print(f"Valid rows: {valid_count}")
    print(f"Invalid rows: {invalid_count}")
    print(f"Warning rows: {warning_count}")

    if invalid_rows:
        print("Invalid samples must be fixed before training handoff.")
        show_table(
            [
                {
                    "base_name": row["base_name"],
                    "image_name": row["image_name"],
                    "label_name": row["label_name"],
                    "errors": row["errors"],
                }
                for row in invalid_rows
            ]
        )

    if warning_rows:
        print("Warnings should be reviewed before training handoff.")
        show_table(
            [
                {
                    "base_name": row["base_name"],
                    "image_name": row["image_name"],
                    "label_name": row["label_name"],
                    "warnings": row["warnings"],
                }
                for row in warning_rows
            ]
        )


## Handoff Checklist

Before sending the dataset to a teammate:

- `invalid_samples.csv` should be empty or every row should be intentionally accepted.
- `warning_samples.csv` should be reviewed.
- `class_counts.csv` should have reasonable balance for `M014`, `M015`, `P004`, and `W011`.
- `NEG_*` images should have empty labels or no sign labels.
- `MIXED_*` images should contain at least two sign classes.
- Image filenames and label filenames should share the same stem.
- No training, splitting, augmentation, or app code changes are performed by this notebook.
